In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
import tensorflow as tf
from tensorflow.keras.layers import Input, Reshape, Dropout, Dense 
from tensorflow.keras.layers import Flatten, BatchNormalization
from tensorflow.keras.layers import Activation, ZeroPadding2D
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.layers import UpSampling2D, Conv2D
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.optimizers import Adam
import numpy as np
from PIL import Image
from tqdm import tqdm
import os 
import time
import matplotlib.pyplot as plt

In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
# Generation resolution - Must be square 
# Training data is also scaled to this.
# Note GENERATE_RES 4 or higher  
# will blow Google CoLab's memory and have not
# been tested extensivly.
GENERATE_RES = 2 # Generation resolution factor 
# (1=32, 2=64, 3=96, 4=128, etc.)
GENERATE_SQUARE = 32 * GENERATE_RES # rows/cols (should be square)
IMAGE_CHANNELS = 3

# Preview image 
PREVIEW_ROWS = 1
PREVIEW_COLS = 1
PREVIEW_MARGIN = 16

# Size vector to generate images from
SEED_SIZE = 100

# Configuration
DATA_PATH = 'data'
EPOCHS = 5000
BATCH_SIZE = 32
BUFFER_SIZE = 60000

print(f"Will generate {GENERATE_SQUARE}px square images.")

Will generate 64px square images.


In [3]:
# --- [CELL 2]: ---
# cell_state: edited
# execution_status: {'status': 'error', 'done': True, 'execution_count': 3}
# === BEFORE (original) ===
# training_binary_path = os.path.join(DATA_PATH,
#         f'training_data_{GENERATE_SQUARE}_{GENERATE_SQUARE}.npy')
# 
# print(f"Looking for file: {training_binary_path}")
# 
# if not os.path.isfile(training_binary_path):
#   start = time.time()
#   print("Loading training images...")
# 
#   training_data = []
#   faces_path = 'data/apple_disease_classification/Train/Blotch_Apple'
#   for filename in tqdm(os.listdir(faces_path)):
#       path = os.path.join(faces_path,filename)
#       image = Image.open(path).resize((GENERATE_SQUARE,
#             GENERATE_SQUARE),Image.LANCZOS)
#       training_data.append(np.asarray(image))
#   training_data = np.reshape(training_data,(-1,GENERATE_SQUARE,
#             GENERATE_SQUARE,3))
#   training_data = training_data.astype(np.float32)
#   training_data = training_data / 127.5 - 1.
# 
# 
#   print("Saving training image binary...")
# #   np.save(training_binary_path,training_data)
#   elapsed = time.time()-start
#   
# else:
#   print("Loading previous training pickle...")
#   training_data = np.load(training_binary_path)

# === AFTER (edited) ===
training_binary_path = os.path.join(DATA_PATH,
        f'training_data_{GENERATE_SQUARE}_{GENERATE_SQUARE}.npy')

print(f"Looking for file: {training_binary_path}")

if not os.path.exists(training_binary_path):
  start = time.time()
  print("Loading training images...")

  training_data = []
  faces_path = 'data/apple_disease_classification/Train/Blotch_Apple'
  
  # Ensure faces_path exists
  if not os.path.exists(faces_path):
      raise FileNotFoundError(f"Training data path not found: {faces_path}")
  
  for filename in tqdm(os.listdir(faces_path)):
      path = os.path.join(faces_path,filename)
      image = Image.open(path).resize((GENERATE_SQUARE,
            GENERATE_SQUARE),Image.LANCZOS)
      training_data.append(np.asarray(image))
  training_data = np.reshape(training_data,(-1,GENERATE_SQUARE,
            GENERATE_SQUARE,3))
  training_data = training_data.astype(np.float32)
  training_data = training_data / 127.5 - 1.


  print("Saving training image binary...")
  # Create directory if it doesn't exist
  os.makedirs(os.path.dirname(training_binary_path), exist_ok=True)
  np.save(training_binary_path, training_data)
  
  elapsed = time.time()-start
  print(f"Saved training binary in {elapsed:.2f} seconds")
else:
  print("Loading previous training pickle...")
  training_data = np.load(training_binary_path)

Looking for file: data/training_data_64_64.npy
Loading training images...


100%|██████████| 116/116 [00:00<00:00, 179.87it/s]


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 3 dimensions. The detected shape was (116, 64, 64) + inhomogeneous part.

In [4]:
import os
import numpy as np
from PIL import Image

# Contract: keep only images that are already (H, W, C) after resize.
faces_path = "data/apple_disease_classification/Train/Blotch_Apple"

expected_valid = 0
for filename in os.listdir(faces_path):
    path = os.path.join(faces_path, filename)
    arr = np.asarray(Image.open(path).resize((GENERATE_SQUARE, GENERATE_SQUARE), Image.LANCZOS))
    if arr.shape == (GENERATE_SQUARE, GENERATE_SQUARE, IMAGE_CHANNELS):
        expected_valid += 1

assert training_data.ndim == 4
assert training_data.shape[1] == GENERATE_SQUARE
assert training_data.shape[2] == GENERATE_SQUARE
assert training_data.shape[3] == IMAGE_CHANNELS
assert training_data.shape[0] == expected_valid, "Patch should filter invalid shapes"

# Basic sanity on produced tensor
assert training_data.shape[0] > 0
assert np.isfinite(training_data).all()
assert training_data.min() >= -1.0 - 1e-6 and training_data.max() <= 1.0 + 1e-6

AttributeError: 'list' object has no attribute 'ndim'